# Q4 Pulse 3: Split pasted AI conversations into speaker turns

The pasted conversations have no speaker labels, and in the export each original line break shows up as a double space. With about 20 pastes, the speakers are coded by hand, and this notebook handles the tedious parts.

**Workflow**

| Step | Section | What happens |
|---|---|---|
| 1 | 3-5 | Each paste is cut into numbered **units** at the double spaces (or line breaks), and every unit gets a **suggested speaker** |
| 2 | 6 | The units are written to `coding_sheet.xlsx`, one row per unit |
| 3 | by hand | Save a copy as `coding_sheet_coded.xlsx`, read down each paste and fix `speaker_final` where the suggestion is wrong |
| 4 | 7-8 | The coded sheet is read back, consecutive units from the same speaker are merged into one turn, and the long file is saved |

**Codes for `speaker_final`:** `P` = participant, `AI` = AI, `X` = exclude (stray text that belongs to neither speaker, such as a leftover button label).

**Input:** `data/processed/00_Q4_3_renamed.csv`

**Outputs**

| File | Contents |
|---|---|
| `output/01_Q4_3_split-conversations/coding_sheet.xlsx` | One row per unit with the suggested speaker (regenerated each time Section 6 runs) |
| `output/01_Q4_3_split-conversations/coding_sheet_coded.xlsx` | **Your** coded copy (this notebook never writes to this file) |
| `data/processed/01_Q4_3_convo_long.csv` | One row per turn: `response_id`, `turn_index`, `exchange_index`, `speaker`, `text` |
| `data/processed/01_Q4_3_convo_summary.csv` | One row per respondent: turn counts, characters per speaker, first prompt |

## 0. Packages

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

# import name : install name
REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
}

missing = [pip_name for mod, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    importlib.invalidate_caches()

for mod in REQUIRED_PACKAGES:
    module = importlib.import_module(mod)
    print(f"  {mod:<10} {getattr(module, '__version__', '')}")

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 300)

## 1. Paths and data

In [ ]:
PROJECT_ROOT = Path(r"C:\path\to\Q4_3_Pulse")

SCRIPT_NAME = "01_Q4_3_split-conversations"
IN_FILE = PROJECT_ROOT / "data" / "processed" / "00_Q4_3_renamed.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR = PROJECT_ROOT / "output" / SCRIPT_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

CODING_SHEET = OUT_DIR / "coding_sheet.xlsx"          # written by Section 6
CODED_SHEET = OUT_DIR / "coding_sheet_coded.xlsx"     # your hand-coded copy, read by Section 7

assert IN_FILE.exists(), f"Renamed file not found: {IN_FILE}"
print(PROJECT_ROOT)

In [ ]:
ID_COL = "response_id"
TEXT_COL = "convo_text"
CARRY_COLS = ["detail_chatbot", "convo_about"]   # copied onto the coding sheet and the long file

df = pd.read_csv(IN_FILE, dtype=str, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
convos = df[[ID_COL, TEXT_COL, *CARRY_COLS]].copy()

has_text = convos[TEXT_COL].fillna("").str.strip() != ""
print(f"{len(convos)} respondents, {has_text.sum()} with pasted text")
print("No pasted text (screenshot only or blank):", convos.loc[~has_text, ID_COL].tolist())

## 2. Settings to tweak

**Splitting.** `UNIT_SPLIT` is the pattern that cuts a paste into units. The default cuts at any line break, or at two or more spaces in a row. If a participant happened to type a double space inside their own message, that message gets cut into two units. Code both units `P` and they are merged back together in Section 7.

**Suggested speaker.** Each unit gets a score built from the cues below. A positive weight points to the participant and a negative weight points to the AI. A unit whose total reaches `P_THRESHOLD` is suggested as `P`, and everything else as `AI`. The coding sheet shows each unit's score and the cues behind the score, so you can see which cues help and adjust the weights.

In [ ]:
UNIT_SPLIT = r"\n+|[ \t]{2,}"

# Lines from the chat app rather than either speaker (whole unit must match; regular expressions)
UI_NOISE = [r"copy", r"copy code", r"edit", r"retry", r"share", r"regenerate",
            r"show thinking", r"show drafts", r"sources?", r"\d+\s*/\s*\d+",
            r".*can make mistakes.*"]

# Words that often open a participant's message
QUESTION_STARTS = ["what", "whats", "what's", "how", "why", "when", "where", "who", "which",
                   "can", "could", "should", "would", "will", "is", "are", "do", "does", "did",
                   "i", "i'm", "im", "my", "me", "tell", "explain", "help", "please", "ok", "okay",
                   "thanks", "thank", "yes", "no", "so", "and", "but"]

STOPWORDS = set("""a an the and or but if of to in on at for with by from as is are was were be been
it its this that these those i you he she they we me my your our their what how why when where who
which can could should would will do does did not no yes so than then there here about into up out
""".split())

CUE_WEIGHTS = {
    "first_unit":          4,   # pastes almost always start with the participant
    "starts_lowercase":    2,   # people type lowercase; AI text is capitalized
    "question_start":      1,   # opens with a word from QUESTION_STARTS
    "ends_with_question":  1,   # ends in "?"
    "after_ai_question":   2,   # the unit before ends in "?" (AI asked a follow-up)
    "echoed_next":         2,   # next unit repeats this unit's content words
    "short":               1,   # shorter than SHORT_CHARS
    "no_end_punct":        1,   # no . ! ? : at the end
    "long":               -4,   # longer than LONG_CHARS
    "header_or_label":    -3,   # Title Case header, or "Something: ..." label
    "in_list_run":        -3,   # between two other short, unpunctuated units (list item)
    "full_sentences":     -2,   # starts with a capital letter and ends with . ! or :
    "before_lowercase":   -3,   # next unit starts lowercase (likely the participant), so this one is likely AI
}

P_THRESHOLD = 4
SHORT_CHARS = 80
LONG_CHARS = 250
ECHO_MIN_SHARED = 1             # shared content words needed for "echoed_next"

## 3. Cut pastes into units

In [ ]:
NOISE_RE = re.compile(rf"^(?:{'|'.join(UI_NOISE)})$", re.IGNORECASE)


def split_units(text):
    """Cut one paste into a list of non-empty units, dropping app button text."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return []
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[\u00a0\u202f\u2007]", " ", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    units = [u.strip() for u in re.split(UNIT_SPLIT, text)]
    return [u for u in units if u and not NOISE_RE.match(u)]

## 4. Suggest a speaker for each unit

In [ ]:
def _words(s):
    return re.findall(r"[a-z0-9']+", s.lower())


def _content_words(s):
    return {w for w in _words(s) if w not in STOPWORDS and len(w) > 2}


def _is_short_unpunct(u):
    return u is not None and len(u) < SHORT_CHARS and not re.search(r"[.!?:]$", u)


def _looks_like_header(u):
    words = u.split()
    if re.match(r"^[A-Z][^:]{0,60}:\s", u):                     # "Type 1 Diabetes: An autoimmune..."
        return True
    caps = sum(w[:1].isupper() for w in words if w.isalpha() and w.lower() not in STOPWORDS)
    content = sum(1 for w in words if w.isalpha() and w.lower() not in STOPWORDS)
    return len(words) <= 8 and content >= 2 and caps == content and not u.endswith("?")


def score_unit(units, i):
    """Return (score, list of cues that fired) for units[i]."""
    u = units[i]
    prev_u = units[i - 1] if i > 0 else None
    next_u = units[i + 1] if i + 1 < len(units) else None
    first_word = (_words(u) or [""])[0]

    cues = {
        "first_unit": i == 0,
        "starts_lowercase": u[:1].islower(),
        "question_start": first_word in QUESTION_STARTS,
        "ends_with_question": u.endswith("?"),
        "after_ai_question": prev_u is not None and prev_u.endswith("?") and len(prev_u) > SHORT_CHARS,
        "echoed_next": next_u is not None and len(_content_words(u) & _content_words(next_u)) >= ECHO_MIN_SHARED,
        "short": len(u) < SHORT_CHARS,
        "no_end_punct": not re.search(r"[.!?:]$", u),
        "long": len(u) > LONG_CHARS,
        "header_or_label": _looks_like_header(u),
        "in_list_run": _is_short_unpunct(u) and _is_short_unpunct(prev_u) and _is_short_unpunct(next_u)
                       and not u[:1].islower(),
        "full_sentences": u[:1].isupper() and bool(re.search(r"[.!:]$", u)),
        "before_lowercase": next_u is not None and next_u[:1].islower(),
    }
    fired = [k for k, v in cues.items() if v]
    return sum(CUE_WEIGHTS[k] for k in fired), fired


def suggest_speakers(units):
    rows = []
    for i in range(len(units)):
        score, fired = score_unit(units, i)
        rows.append({"score": score,
                     "suggested": "P" if score >= P_THRESHOLD else "AI",
                     "cues": ", ".join(fired)})
    return rows

### 4a. Try the suggestions on a made-up paste

The example below is invented, in the same shape as the real pastes (double spaces where the line breaks were). Edit it or add your own to see how the cues respond before building the sheet.

In [ ]:
EXAMPLE = (
    "is it bad to take ibuprofen every day  Taking ibuprofen every day can be safe for some people, "
    "but long-term daily use carries real risks, so it is worth checking with a clinician.  "
    "Possible Risks  Stomach irritation  Ulcers or bleeding  Kidney strain  Higher blood pressure  "
    "Lower-Risk Habits: Take it with food, use the lowest dose that works, and avoid combining it with other NSAIDs.  "
    "Is the pain you are treating ongoing, or is this for something short-term?  "
    "its for my back pain from work  For ongoing back pain, daily ibuprofen is usually not the best long-term plan. "
    "A doctor or physical therapist can help find the cause."
)

units = split_units(EXAMPLE)
pd.DataFrame(suggest_speakers(units)).assign(unit=units)[["unit", "suggested", "score", "cues"]]

## 5. Build the unit table for all pastes

In [ ]:
def build_units(convos):
    rows = []
    for _, r in convos.iterrows():
        units = split_units(r[TEXT_COL])
        for i, (u, s) in enumerate(zip(units, suggest_speakers(units)), start=1):
            rows.append({ID_COL: r[ID_COL], **{c: r[c] for c in CARRY_COLS},
                         "unit_index": i, "n_chars": len(u), "unit_text": u,
                         "suggested": s["suggested"], "score": s["score"], "cues": s["cues"]})
    out = pd.DataFrame(rows)
    out["speaker_final"] = out["suggested"]
    out["notes"] = ""
    return out


units_df = build_units(convos)
print(f"{len(units_df)} units from {units_df[ID_COL].nunique()} pastes")
units_df.groupby(ID_COL).agg(n_units=("unit_index", "max"),
                             n_suggested_P=("suggested", lambda s: (s == "P").sum()))

## 6. Write the coding sheet

The sheet is formatted for reading down a paste: text wraps, pastes alternate shading, suggested `P` rows are bold, and `speaker_final` has a P / AI / X dropdown.

**Save your coded version as `coding_sheet_coded.xlsx`.** Rerunning this section overwrites `coding_sheet.xlsx` but never touches your coded copy.

In [ ]:
def write_coding_sheet(units_df, path):
    cols = [ID_COL, *CARRY_COLS, "unit_index", "unit_text", "suggested", "speaker_final",
            "notes", "score", "cues", "n_chars"]
    units_df[cols].to_excel(path, index=False, sheet_name="units")

    wb = load_workbook(path)
    ws = wb["units"]
    widths = {ID_COL: 12, "unit_index": 7, "unit_text": 90, "suggested": 10,
              "speaker_final": 13, "notes": 25, "score": 7, "cues": 40, "n_chars": 8}
    header = [c.value for c in ws[1]]
    for j, name in enumerate(header, start=1):
        ws.column_dimensions[ws.cell(1, j).column_letter].width = widths.get(name, 12)
        ws.cell(1, j).font = Font(bold=True)
    ws.freeze_panes = "A2"

    shade = PatternFill("solid", fgColor="EEF2F7")
    final_fill = PatternFill("solid", fgColor="FFF4D6")
    text_col = header.index("unit_text") + 1
    final_col = header.index("speaker_final") + 1
    sugg_col = header.index("suggested") + 1

    current, band = None, False
    for row in ws.iter_rows(min_row=2):
        rid = row[0].value
        if rid != current:
            current, band = rid, not band
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            if band:
                cell.fill = shade
        row[final_col - 1].fill = final_fill
        if row[sugg_col - 1].value == "P":
            row[text_col - 1].font = Font(bold=True)

    dv = DataValidation(type="list", formula1='"P,AI,X"', allow_blank=False)
    ws.add_data_validation(dv)
    letter = ws.cell(1, final_col).column_letter
    dv.add(f"{letter}2:{letter}{ws.max_row}")
    wb.save(path)


write_coding_sheet(units_df, CODING_SHEET)
print(f"Wrote {CODING_SHEET}")
print(f"Save your coded copy as {CODED_SHEET.name}")

## 7. Read the coded sheet back

Checks that every unit has a valid code and that the sheet still matches the pastes, then merges consecutive units from the same speaker into turns. Units coded `X` are dropped. Merged units are joined with a line break, which restores the original line structure.

In [ ]:
assert CODED_SHEET.exists(), f"Code the sheet first, then save it as {CODED_SHEET.name}"

coded = pd.read_excel(CODED_SHEET, dtype={ID_COL: str})
coded["speaker_final"] = coded["speaker_final"].astype(str).str.strip().str.upper()

bad = coded[~coded["speaker_final"].isin(["P", "AI", "X"])]
assert bad.empty, f"{len(bad)} unit(s) without a valid P / AI / X code:\n{bad[[ID_COL, 'unit_index', 'speaker_final']]}"

# The coded sheet should cover the same units as the current pastes
check = coded[[ID_COL, "unit_index", "unit_text"]].merge(
    units_df[[ID_COL, "unit_index", "unit_text"]], on=[ID_COL, "unit_index"], how="outer",
    suffixes=("_coded", "_now"), indicator=True)
mismatch = check[(check["_merge"] != "both") | (check["unit_text_coded"] != check["unit_text_now"])]
if len(mismatch):
    print(f"Warning: {len(mismatch)} unit(s) differ between the coded sheet and the current split. "
          "This happens if UNIT_SPLIT or UI_NOISE changed after coding.")
    display(mismatch.head(20))
else:
    print("Coded sheet matches the current split.")

# How often were the suggestions right? (useful if later pulses need an automated split)
agree = (coded["suggested"] == coded["speaker_final"]).mean()
print(f"Suggested speaker matched your code for {agree:.0%} of units")
pd.crosstab(coded["suggested"], coded["speaker_final"], margins=True)

## 8. Build the long file and save

In [ ]:
SPEAKER_NAMES = {"P": "participant", "AI": "ai"}


def units_to_turns(coded):
    kept = coded[coded["speaker_final"] != "X"].sort_values([ID_COL, "unit_index"])
    records = []
    for rid, g in kept.groupby(ID_COL, sort=False):
        carry = {c: g[c].iloc[0] for c in CARRY_COLS if c in g}
        run_id = (g["speaker_final"] != g["speaker_final"].shift()).cumsum()
        exchange = 0
        for turn_index, (_, run) in enumerate(g.groupby(run_id, sort=False), start=1):
            speaker = SPEAKER_NAMES[run["speaker_final"].iloc[0]]
            if speaker == "participant":
                exchange += 1
            text = "\n".join(run["unit_text"].astype(str))
            records.append({ID_COL: rid, **carry, "turn_index": turn_index,
                            "exchange_index": exchange or np.nan, "speaker": speaker,
                            "unit_first": run["unit_index"].min(), "unit_last": run["unit_index"].max(),
                            "n_units": len(run), "text": text,
                            "n_chars": len(text), "n_words": len(text.split())})
    return pd.DataFrame(records)


convo_long = units_to_turns(coded)

convo_summary = (convo_long.groupby(ID_COL)
    .agg(n_turns=("turn_index", "max"),
         n_participant=("speaker", lambda s: (s == "participant").sum()),
         n_ai=("speaker", lambda s: (s == "ai").sum()),
         chars_participant=("n_chars", lambda s: s[convo_long.loc[s.index, "speaker"] == "participant"].sum()),
         chars_ai=("n_chars", lambda s: s[convo_long.loc[s.index, "speaker"] == "ai"].sum()),
         starts_with=("speaker", "first"))
    .reset_index()
    .merge(convo_long.query("speaker == 'participant'").groupby(ID_COL)["text"].first()
           .rename("first_prompt").reset_index(), on=ID_COL, how="left"))

convo_long.to_csv(PROCESSED_DIR / "01_Q4_3_convo_long.csv", index=False, encoding="utf-8-sig")
convo_summary.to_csv(PROCESSED_DIR / "01_Q4_3_convo_summary.csv", index=False, encoding="utf-8-sig")
print(f"long file: {len(convo_long)} turns from {convo_long[ID_COL].nunique()} pastes")
convo_summary